In [ ]:
# spatial_sdm.py
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

import os
from pathlib import Path
import numpy as np
import pandas as pd

import pytensor.tensor as pt

# ------------------------------------------------------------
# 0) Define Data Path
# ------------------------------------------------------------
try:
    # 1. Preferred method for scripts (.py files):
    # Get the directory of the current script file.
    BASE_DIR = Path(file).parent
except NameError:
    # 2. Fallback for notebooks (Colab/Jupyter):
    # Use the current working directory, which should be the Git root
    # after cloning and changing directory (see Colab setup below).
    BASE_DIR = Path(os.getcwd())

# Define the relative path to the raw data folder
RAW_DATA_PATH = BASE_DIR / "data"

# Sanity check (Optional, but useful for debugging)
if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"The data/raw directory was not found at the expected path: {RAW_DATA_PATH}"
    )

print(f"Data will be loaded from: {RAW_DATA_PATH}")

Data will be loaded from: /content/Silk-Road-Tourism-Spillovers-Spatial-Bayesian-Durbin-Model/data


In [ ]:
import spatial_sdm as sdm

/usr/local/lib/python3.12/dist-packages/arviz/__init__.py:39: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [ ]:
# ============================================================
# Spatial Durbin Model (SDM) – Main Execution Script
# ============================================================

# spatial_sdm.py
"""
Spatial SDM (Spatial Durbin Model) for panel data in PyMC.
...
"""

from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import os


# ------------------------------------------------------------
# 0) Define Data Path
# ------------------------------------------------------------
try:
    # 1. Preferred method for scripts (.py files):
    # Get the directory of the current script file.
    BASE_DIR = Path(__file__).parent
except NameError:
    # 2. Fallback for notebooks (Colab/Jupyter):
    # Use the current working directory, which should be the Git root
    # after cloning and changing directory (see Colab setup below).
    BASE_DIR = Path(os.getcwd())

# Define the relative path to the raw data folder
RAW_DATA_PATH = BASE_DIR / "data"

# Sanity check (Optional, but useful for debugging)
if not RAW_DATA_PATH.is_dir():
    raise FileNotFoundError(
        f"The data/raw directory was not found at the expected path: {RAW_DATA_PATH}"
    )

print(f"Data will be loaded from: {RAW_DATA_PATH}")

# ------------------------------------------------------------
# 1) Load data
# ------------------------------------------------------------
# ------------------------------------------------------------
# Use the RAW_DATA_PATH variable and the / operator from pathlib
# to construct the full file path.

df_sorted = pd.read_excel(RAW_DATA_PATH / "df_sorted.xlsx")
W_cul04_raw = pd.read_excel(RAW_DATA_PATH / "W_cul04.xlsx")
W_cul06_raw = pd.read_excel(RAW_DATA_PATH / "W_cul06.xlsx")
W_cul05_raw = pd.read_excel(RAW_DATA_PATH / "W_cul05.xlsx")
W_geo_raw = pd.read_excel(RAW_DATA_PATH / "W_geo.xlsx")
W_trade_raw = pd.read_excel(RAW_DATA_PATH / "W_trade.xlsx")


# ------------------------------------------------------------
# 2) Clean spatial weights and align with panel data
# ------------------------------------------------------------
W_cul04 = sdm.prepare_W_from_excel(W_cul04_raw)
W_cul06 = sdm.prepare_W_from_excel(W_cul06_raw)
W_cul05 = sdm.prepare_W_from_excel(W_cul05_raw)
W_geo = sdm.prepare_W_from_excel(W_geo_raw)
W_trade = sdm.prepare_W_from_excel(W_trade_raw)


# ** IMPORTANT: If you want to assign a different weight matrix for this execution,
# you should change the weights matrices below to the desired ones. **
# This can be done by modifying the corresponding W_* variables.

df, W = sdm.align_df_and_W(df_sorted, W_trade)

# Ensure correct sorting (VERY IMPORTANT)
df = df.sort_values(["year", "country"]).copy()
df["year"] = df["year"].astype(int)

# ------------------------------------------------------------
# 3) Sanity checks
# ------------------------------------------------------------
years = sorted(df["year"].unique())
countries = sorted(df["country"].unique())

print("Number of countries (N):", len(countries))
print("Number of years (T):", len(years))
print("Number of observations (NT):", df.shape[0])
print("Expected NT = N * T:", len(countries) * len(years))
print("Balanced panel:", df.shape[0] == len(countries) * len(years))
print("W shape:", W.shape)

# ------------------------------------------------------------
# 4) Fit SDM on the full dataset
# ------------------------------------------------------------



trace_full, countries_sorted, years_sorted, W_base = sdm.run_sdm_model(
    df, W,
    draws=1000,
    tune=1000,
    chains=4,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=True,
)

print("SDM estimation completed.")

# ------------------------------------------------------------
# 5) (Optional) Save posterior samples for reproducibility
# ------------------------------------------------------------
import arviz as az
az.to_netcdf(trace_full, "trace_sdm_full.nc")


 Progress                    Draws   Divergences   Step size   Grad evals   Sampling Speed   Elapsed   Remaining  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━━━━━━━━━━━━━━━━━━━   2000    0             0.004       2047         3.10 draws/s     0:10:45   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   2000    0             0.004       2047         1.57 draws/s     0:21:10   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   2000    0             0.003       2047         1.04 draws/s     0:32:05   0:00:00    
  ━━━━━━━━━━━━━━━━━━━━━━━━━   2000    0             0.003       1279         1.27 s/draws     0:42:10   0:00:00

SDM estimation completed.


'trace_sdm_full.nc'

In [ ]:
import importlib
import spatial_sdm as sdm
importlib.reload(sdm)

summary_main, df_alpha, df_time_alpha = sdm.summarize_sdm_trace(
    trace_full, countries_sorted, years_sorted, round_to=4, verbose=True
)

summary_main
df_alpha.head()
df_time_alpha.head()


Main coefficients summary:
             mean      sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   ess_bulk  \
rho      -0.5698  0.1655 -0.8902  -0.2780     0.0032   0.0025  2658.5879   
beta[0]   0.7015  0.1473  0.4196   0.9693     0.0027   0.0023  2965.8091   
beta[1]   0.2233  0.0848  0.0675   0.3857     0.0015   0.0013  3275.2049   
beta[2]   0.3022  0.0842  0.1402   0.4616     0.0018   0.0014  2229.0833   
beta[3]   0.5734  0.1111  0.3475   0.7667     0.0017   0.0015  4117.0692   
gamma[0]  1.7973  0.2418  1.3450   2.2511     0.0047   0.0035  2701.6208   
gamma[1] -0.2345  0.3128 -0.8145   0.3549     0.0049   0.0053  4130.0285   
gamma[2]  1.3852  0.2525  0.9153   1.8731     0.0049   0.0040  2635.7147   
gamma[3]  2.8276  0.4684  1.9490   3.6914     0.0082   0.0074  3248.4658   
sigma     0.2505  0.0143  0.2243   0.2784     0.0003   0.0003  3264.8226   

           ess_tail   r_hat  
rho       2238.0727  1.0012  
beta[0]   2979.2751  0.9999  
beta[1]   2778.1624  1.0000  
beta[2]   2066.

,year,time_alpha_mean,time_alpha_sd
0,2002,0.023850,0.478084
1,2003,-0.214490,0.469410
2,2004,0.032504,0.474110
3,2005,-0.096631,0.474857
4,2006,0.281778,0.478123


In [ ]:
import numpy as np
import pandas as pd
#import arviz as az
import spatial_sdm as sdm

X, WX, y, country_idx, year_idx, countries_sorted, years_sorted, N, T = sdm.build_design_mats(df, W_base)

y_hat = sdm.posterior_mean_predictions(trace_full, X, WX, country_idx, year_idx, W_base, N=N, T=T)

mse_by_country = {}
for c_i, c in enumerate(countries_sorted):
    mask = (country_idx == c_i)
    mse_by_country[c] = float(np.mean((y[mask] - y_hat[mask])**2))

mse_df = pd.DataFrame({"country": list(mse_by_country.keys()),
                       "mse": list(mse_by_country.values())}).sort_values("mse")

print("MSE by country (lower is better):")
display(mse_df)


# Optional: save
mse_df.to_csv("mse_by_country.csv", index=False)


# ------------------------------------------------------------
# 3) Moran's I on SDM residuals by year
# ------------------------------------------------------------
# We need a PySAL weights object + residuals per year
from libpysal.weights import W as W_pysal
from esda.moran import Moran


morans_df = sdm.morans_I_by_year(
    trace=trace_full,
    X=X,
    WX=WX,
    y=y,
    country_idx=country_idx,
    year_idx=year_idx,
    W_base=W_base,
    years_sorted=years_sorted,
    N=N,
    T=T,
)

display(morans_df)
morans_df.to_csv("moransI_residuals_by_year.csv", index=False)


MSE by country (lower is better):


,country,mse
9,Saudi Arabia,0.019237
10,Türkiye,0.028799
2,Egypt,0.034855
6,Italy,0.035424
4,India,0.036838
5,Iran,0.038564
7,Kazakhstan,0.041627
3,Georgia,0.049705
0,Azerbaijan,0.061790
1,China,0.070109


,year,moran_I,p_norm,p_sim
0,2002,-0.066241,0.756214,0.393
1,2003,-0.081071,0.861809,0.447
2,2004,0.039506,0.199522,0.112
3,2005,-0.124909,0.818818,0.379
4,2006,0.129160,0.035084,0.017
5,2007,0.106665,0.057365,0.021
6,2008,-0.042279,0.595551,0.310
7,2009,-0.189192,0.412091,0.200
8,2010,0.008932,0.316461,0.149
9,2011,-0.039615,0.578681,0.290


In [ ]:
# ------------------------------------------------------------
# Leave-One-Country-Out (LOCO) cross-validation
# ------------------------------------------------------------
# NOTE: This is computationally expensive.
# Start with small draws/tune for testing.

res_loco = sdm.loco_cv(
    df=df,
    W_df=W,
    draws=1000,           # increase after testing
    tune=1000,
    chains=2,
    target_accept=0.95,
    random_seed=123,
    cores=None,
    progressbar=False,
)

print(res_loco["status"].value_counts())
print(res_loco.head())

# Save LOCO results
res_loco.to_csv("loco_results.csv", index=False)

print("LOCO cross-validation completed.")
